In [ ]:
#https://ryan-reilly.medium.com/gridsearch-pipelines-of-multiple-models-on-multiclass-classification-e9124b6ea2e3
#https://scikit-learn.org/stable/tutorial/statistical_inference/putting_together.html

#Feature selection
#https://scikit-learn.org/stable/modules/feature_selection.html

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
abandono = pd.read_csv('./Abandono/abandono.csv',index_col=0)
abandono

In [ ]:
grades_per_exam = pd.read_csv('./Grades/grades_per_exam.csv',index_col=0)
grades_per_exam

# Pipeline + GridSearchCV


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn import svm

In [ ]:
import sklearn
sklearn.metrics.get_scorer_names()

In [ ]:
def abandono_pipeline(w, indicadores):
  theta = pd.read_csv('./Modelos/Cummulative/Configuración all students/theta_allstudents_week{}.csv'.format(w),index_col=0)
  w_ind = ind.loc[theta.index]
  w_ind['theta'] = theta['conf1']
  w_ind['passed'] = abandono['curso_superado']
  w_ind['dropped_out'] = abandono['abandono']

  w_ind['CFA'] = w_ind['CFA'].replace('0 problems attempted',0)
  w_ind['avg_grade'] = w_ind['avg_grade'].replace('0 problems attempted',0)
  w_ind['avg_attempts'] = w_ind['avg_attempts'].replace('0 problems attempted',0)
  w_ind['dropped_out'] = w_ind['dropped_out'].replace('No',0).replace('Sí',1)
  w_ind['passed'] = w_ind['passed'].replace('No',0).replace('Sí',1)

  X = w_ind.drop(['dropped_out','passed'],axis=1)
  y = w_ind['dropped_out']

  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

  pipe_lr = Pipeline([('scl', StandardScaler()),
                    ('LR', LogisticRegression(random_state=42))])
  pipe_dt = Pipeline([('scl', StandardScaler()),
                    ('DT',DecisionTreeClassifier(random_state=42))])
  pipe_rf = Pipeline([('scl', StandardScaler()),
                      ('RF',RandomForestClassifier(random_state=42))])
  pipe_knn = Pipeline([('scl', StandardScaler()),
                      ('KNN', KNeighborsClassifier())])
  pipe_svm = Pipeline([('scl', StandardScaler()),
                      ('SVM', svm.SVC(random_state=42))])
  pipe_xgb = Pipeline([('scl', StandardScaler()),
                      ('XGB', XGBClassifier(random_state=42))])

  param_range = [1, 2, 3, 5]
  param_range_fl = [1.0, 0.5, 0.1]
  n_estimators = [50,75,100]
  learning_rates = [.1,.2,.3]
  lr_param_grid = [{'LR__penalty': ['l1', 'l2'],
                   'LR__C': param_range_fl,
                   'LR__solver': ['liblinear']}]
  dt_param_grid = [{'DT__criterion': ['gini', 'entropy'],
                    'DT__min_samples_leaf': param_range,
                    'DT__max_depth': param_range,
                    'DT__min_samples_split': param_range[1:]}]
  rf_param_grid = [{'RF__min_samples_leaf': param_range,
                    'RF__max_depth': param_range,
                    'RF__min_samples_split': param_range[1:]}]
  knn_param_grid = [{'KNN__n_neighbors': param_range,
                    'KNN__weights': ['uniform', 'distance'],
                    'KNN__metric': ['euclidean', 'manhattan']}]
  svm_param_grid = [{'SVM__kernel': ['linear', 'rbf'],
                      'SVM__C': param_range}]
  xgb_param_grid = [{'XGB__learning_rate': learning_rates,
                      'XGB__max_depth': param_range,
                      'XGB__min_child_weight': param_range[:2],
                      'XGB__subsample': param_range_fl,
                      'XGB__n_estimators': n_estimators}]

  lr_grid_search = GridSearchCV(estimator=pipe_lr,
        param_grid=lr_param_grid,
        scoring='roc_auc',
        cv=3)
  dt_grid_search = GridSearchCV(estimator=pipe_dt,
        param_grid=dt_param_grid,
        scoring='roc_auc',
        cv=3)
  rf_grid_search = GridSearchCV(estimator=pipe_rf,
          param_grid=rf_param_grid,
          scoring='roc_auc',
          cv=3)
  knn_grid_search = GridSearchCV(estimator=pipe_knn,
          param_grid=knn_param_grid,
          scoring='roc_auc',
          cv=3)
  svm_grid_search = GridSearchCV(estimator=pipe_svm,
          param_grid=svm_param_grid,
          scoring='roc_auc',
          cv=3)
  xgb_grid_search = GridSearchCV(estimator=pipe_xgb,
          param_grid=xgb_param_grid,
          scoring='roc_auc',
          cv=3)
  grids = [lr_grid_search, dt_grid_search, rf_grid_search, knn_grid_search, svm_grid_search, xgb_grid_search]
  names = ['lr','dt', 'rf', 'knn', 'svm', 'xgb']
  n = 0
  for pipe in grids:
    print(names[n])
    pipe.fit(X_train,y_train)
    n+=1

  aucs = {}
  grid_dict = {0: 'Logistic Regression', 1: 'Decision Trees',
             2: 'Random Forest', 3: 'K-Nearest Neighbors',
             4: 'Support Vector Machines', 5: 'XGBoost'}
  for i, model in enumerate(grids):
    aucs[grid_dict[i]] = model.score(X_test,y_test)

  return aucs

In [ ]:
aucs = {}
abandono = pd.read_csv('./Abandono/abandono.csv',index_col=0)

In [ ]:
from sklearn import preprocessing
lbl = preprocessing.LabelEncoder()

for w in range(1,8):
  print(w)
  indicadores =  pd.read_csv('./Modelos/Cummulative/Indicadores/indicadores_week1_{}.csv'.format(w),index_col=0)
  theta = pd.read_csv('./Modelos/Cummulative/Configuración all students/theta_allstudents_week{}.csv'.format(w),index_col=0)
  w_ind = indicadores.loc[theta.index]
  w_ind['passed'] = abandono['curso_superado']
  w_ind['dropped_out'] = abandono['abandono']


  w_ind['CFA'] = w_ind['CFA'].replace('0 problems attempted',0)
  w_ind['avg_grade'] = w_ind['avg_grade'].replace('0 problems attempted',0)
  w_ind['avg_attempts'] = w_ind['avg_attempts'].replace('0 problems attempted',0)
  w_ind['dropped_out'] = w_ind['dropped_out'].replace('No',0).replace('Sí',1)
  w_ind['passed'] = w_ind['passed'].replace('No',0).replace('Sí',1)
  #w_ind = w_ind.drop(['n_attempted'],axis=1)

  X = w_ind.drop(['dropped_out','passed'],axis=1)
  y = w_ind['dropped_out']
  
  from sklearn import preprocessing


  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
  lbl = preprocessing.LabelEncoder()
  X_train['CFA'] = lbl.fit_transform(X_train['CFA'].astype(str))
  X_train['avg_grade'] = lbl.fit_transform(X_train['avg_grade'].astype(str))
  X_train['avg_attempts'] = lbl.fit_transform(X_train['avg_attempts'].astype(str))
  # fit model no training data
  model = XGBClassifier(learning_rate= 0.1,max_depth= 5,min_child_weight= 1,n_estimators= 75,subsample=1.0)
  model.fit(X_train, y_train, eval_metric='auc')


  theta = pd.read_csv('./Modelos/Cummulative/Configuración al menos un evaluable/theta_one_week{}.csv'.format(w),index_col=0)

  X_test = X.loc[theta.index]
  y_test = y.loc[theta.index]

  X_test['CFA'] = lbl.fit_transform(X_test['CFA'].astype(str))
  X_test['avg_grade'] = lbl.fit_transform(X_test['avg_grade'].astype(str))
  X_test['avg_attempts'] = lbl.fit_transform(X_test['avg_attempts'].astype(str))
  aucs['Week {}'.format(w)] = model.score(X_test,y_test)

In [ ]:
aucs

In [ ]:
abandono = [aucs]
nombres = ['all students']
i = 0
for a in abandono:
  x, y = zip(*a.items())
  plt.plot(x, y, marker='o', label=nombres[i])
  plt.ylim(0.55,1)
  plt.legend(loc='lower right',fontsize='x-small')
  plt.xlabel('Week')
  plt.title('AUC abandono')
  i += 1

# **Nota final**

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor

In [ ]:
rmses = {}
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

In [ ]:
from sklearn import preprocessing
lbl = preprocessing.LabelEncoder()

for w in range(1,8):
  print(w)
  indicadores =  pd.read_csv('./Modelos/Cummulative/Indicadores/indicadores_week1_{}.csv'.format(w),index_col=0)
  theta_all = pd.read_csv('./Modelos/Cummulative/Configuración all students/theta_allstudents_week{}.csv'.format(w),index_col=0)
  theta = pd.read_csv('./Modelos/Cummulative/Configuración all attempted/theta_allatt_week{}.csv'.format(w),index_col=0)
  w_ind = indicadores.loc[theta_all.index]
  w_ind['theta'] = theta['conf1']
  w_ind['grade'] = grades_per_exam['Course grade']
  w_ind['grade'] = w_ind['grade'].fillna(0)


  w_ind['CFA'] = w_ind['CFA'].replace('0 problems attempted',0)
  w_ind['avg_grade'] = w_ind['avg_grade'].replace('0 problems attempted',0)
  w_ind['avg_attempts'] = w_ind['avg_attempts'].replace('0 problems attempted',0)

  X = w_ind.drop(['grade'],axis=1)
  X_2 = X.loc[theta.index]
  print('X len should be 1371',len(X_2))
  y = w_ind['grade']
  y_2 = y.loc[theta.index]
  print('y len should be 1371',len(y_2))
  from sklearn import preprocessing


  X_train, X_test, y_train, y_test = train_test_split(X_2, y_2, test_size=0.25, random_state=42)
  lbl = preprocessing.LabelEncoder()

  X_train['CFA'] = lbl.fit_transform(X_train['CFA'].astype(str))
  X_train['avg_grade'] = lbl.fit_transform(X_train['avg_grade'].astype(str))
  X_train['avg_attempts'] = lbl.fit_transform(X_train['avg_attempts'].astype(str))

  pipe_xgb = Pipeline([('scl', StandardScaler()),
                      ('XGB', XGBRegressor())])

  xgb_param_grid = [{'XGB__learning_rate': [0.1],
                      'XGB__max_depth': [5],
                      'XGB__min_child_weight': [2.0],
                      'XGB__subsample': [1.0],
                      'XGB__n_estimators': [75]}]

  xgb_grid_search = GridSearchCV(estimator=pipe_xgb,
          param_grid=xgb_param_grid,
          scoring='neg_mean_squared_error',
          cv=3)
  grids = [xgb_grid_search]
  names = ['xgb']
  n = 0
  for pipe in grids:
    print(names[n])
    pipe.fit(X_train,y_train)
    n+=1

  theta = pd.read_csv('./Modelos/Cummulative/Configuración al menos un evaluable/theta_one_week{}.csv'.format(w),index_col=0)

  X_test = X.loc[theta.index]
  print('X len should be 8708',len(X_test))
  y_test = y.loc[theta.index]
  print('X len should be 8708',len(y_test))

  X_test['CFA'] = lbl.fit_transform(X_test['CFA'].astype(str))
  X_test['avg_grade'] = lbl.fit_transform(X_test['avg_grade'].astype(str))
  X_test['avg_attempts'] = lbl.fit_transform(X_test['avg_attempts'].astype(str))

  mses = {}
  grid_dict = {0: 'XGBoost'}
  for i, model in enumerate(grids):
    mses = -(model.score(X_test,y_test))


In [ ]:
rmses

In [ ]:
theta = pd.read_csv('./Modelos/Cummulative/Configuración all attempted/theta_allatt_week{}{}.csv'.format(w),index_col=0)


In [ ]:
from math import sqrt
for i in rmses.keys():
  print(rmses[i]*10)

In [ ]:
fig = plt.subplots(2,3, figsize=(12,5),constrained_layout = True)

models = ['Ridge Regression','Decision Trees', 'Random Forest','K-Nearest Neighbors','Support Vector Machines','XGBoost']

mse_model = {}
for m in models:
  mse_model[m] = {}
  for i in range(1,8):
    mse_model[m][i] = mses['Week {}'.format(i)][m]
print(mse_model)

i=0
for m in models:
  x, y = zip(*mse_model[m].items())
  ax = plt.subplot(2, 3, i + 1)
  plt.plot(x, y, marker='o')
  plt.ylim(0,0.02)
  plt.xlabel('Week')
  plt.title(f'AUC {m}')
  i+=1

In [ ]:
mses